In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import h5py
import json
plt.style.use('./graph_preset.mplstyle')

In [2]:
read_path = Path("./0604001150/results.h5")


In [3]:
with h5py.File(read_path, "r") as f: # read_paths[#] that you want to read
    print(f"--- Structure of {read_path} ---")

    def print_structure(name, obj):
        # データセットの場合は形状とデータ型も表示
        if isinstance(obj, h5py.Dataset):
            print(f"  {name} (Dataset) | Shape: {obj.shape}, Dtype: {obj.dtype}")
        # グループの場合はグループ名を表示
        elif isinstance(obj, h5py.Group):
            print(f"  {name} (Group)")

    f.visititems(print_structure)
    print("---------------------------------")

--- Structure of 0604001150\results.h5 ---
  input (Group)
  learning_curve (Group)
  output (Group)
  output/validation_samples (Dataset) | Shape: (100, 17), Dtype: float32
  output/validation_summary (Dataset) | Shape: (1, 5), Dtype: float32
---------------------------------


In [4]:
df_data = dict()

def store_dataset(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"  Loading: {name} | Shape: {obj.shape}")
        df = pd.DataFrame(obj[:])
        df_data[name] = df

def store_dataset(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"  Loading: {name} | Shape: {obj.shape}")

        # columns 属性があれば JSON から復元
        cols_attr = obj.attrs.get("columns", None)
        columns = None
        if cols_attr is not None:
            # 古い h5py だと bytes / np.bytes_ で返ることがある
            if isinstance(cols_attr, (bytes, np.bytes_)):
                cols_attr = cols_attr.decode("utf-8")
            columns = json.loads(cols_attr)

        # DataFrame 化（列名があれば使う）
        data = obj[:]  # dset[:, :] と同じ
        if columns is not None:
            df = pd.DataFrame(data, columns=columns)
        else:
            df = pd.DataFrame(data)

        df_data[name] = df


with h5py.File(read_path, "r") as f:
    print(f"--- Loading all datasets from {read_path} ---")
    f.visititems(store_dataset)
    print("---------------------------------------------")

print("\n--- Dictionary Keys ---")
print(list(df_data.keys()))
print("-----------------------")

--- Loading all datasets from 0604001150\results.h5 ---
  Loading: output/validation_samples | Shape: (100, 17)
  Loading: output/validation_summary | Shape: (1, 5)
---------------------------------------------

--- Dictionary Keys ---
['output/validation_samples', 'output/validation_summary']
-----------------------


In [5]:
pd.set_option('display.max_rows', None)

In [14]:
df_data["output/repeat_2"]

,h1,h2,h3,h4,h5,s1,s2,s3,s4,s5,a,b,k,S11,Metric,gamma,routine_idx,best
0,7.637619,2.351943,8.532845,5.061182,6.935970,1.505953,0.693909,0.725837,1.704291,0.183595,4.580905,3.163502,2.517591,-6.283768,NaN,NaN,0.0,-6.283768
1,4.514539,9.090914,6.890657,3.466976,9.176226,1.870636,1.875921,1.851147,0.024076,1.616310,6.144468,6.525059,5.688117,-5.804924,NaN,NaN,0.0,-6.283768
2,8.385744,4.575850,2.811249,9.448431,6.573482,1.434560,1.212904,0.330059,1.524796,0.606366,5.333173,4.623327,3.085552,-10.621166,NaN,NaN,0.0,-10.621166
3,2.167021,8.297297,6.313925,2.343347,8.826257,1.832917,0.937229,1.452967,1.979948,1.137666,2.299647,4.767825,5.173294,-5.217446,NaN,NaN,0.0,-10.621166
4,5.174329,6.883579,5.946410,9.601224,9.753150,1.977302,0.761415,1.730861,0.165657,1.549495,2.204813,4.322384,4.492930,-4.685142,NaN,NaN,0.0,-10.621166
5,5.515934,6.109837,9.718186,7.792818,3.940467,1.756184,0.427148,0.177711,0.201869,0.320273,4.915356,3.920048,4.627480,-3.873078,NaN,NaN,0.0,-10.621166
6,3.008132,6.545090,3.444530,4.050060,1.989955,1.046733,1.770814,0.224951,0.908558,0.513223,4.275126,5.928197,5.452134,-9.019623,NaN,NaN,0.0,-10.621166
7,9.082014,3.138032,8.805615,1.843302,4.247350,1.585650,0.346964,1.390353,1.101837,1.764401,3.053410,3.405515,1.993851,-0.656620,NaN,NaN,0.0,-10.621166
8,9.915238,9.195186,5.271423,5.914814,7.389038,1.600052,1.938591,1.060708,1.449787,0.904358,3.436585,5.376214,1.679092,-0.791911,NaN,NaN,0.0,-10.621166
9,6.681750,3.584214,9.540702,6.023514,4.992603,1.938973,1.042070,0.565826,1.302861,0.239474,6.988968,2.658395,1.127029,-0.031513,NaN,NaN,0.0,-10.621166


In [15]:
combined = pd.concat(df_data, names=["source"])
min_pos = combined["S11"].idxmin()

print(f"Min S11: {combined.loc[min_pos, 'S11']}")
print(f"Location: {min_pos}")
print("That row:")
print(combined.loc[min_pos])

Min S11: -26.856151580810547
Location: ('output/repeat_3', 36)
That row:
h1               6.226182
h2               8.794176
h3               3.161295
h4               8.149709
h5               7.147060
s1               1.003729
s2               1.371851
s3               1.886560
s4               0.923806
s5               1.245569
a                2.031578
b                3.560018
k                4.278458
S11            -26.856152
Metric                NaN
gamma          504.611938
routine_idx     17.000000
best           -26.856152
Name: (output/repeat_3, 36), dtype: float32
